<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

## FLOPS Analysis

- FLOPs（Floating Point Operations，浮点运算次数）表示完成一次计算需要执行多少次浮点运算，可用于估算神经网络的计算复杂度
- FLOP/s（Floating Point Operations Per Second，每秒浮点运算次数）才表示硬件的计算速度
- 在输入形状相同的条件下，FLOPs 越高通常意味着模型需要更多计算资源和能量

In [ ]:
# 首次运行前，如果缺少 THOP 等额外依赖，请在终端中执行下面这条命令
# pip install -r requirements-extra.txt

In [1]:
# version() 用于读取当前 Python 环境中已安装包的版本号
from importlib.metadata import version

import matplotlib  # 本 notebook 不绘图；导入它可以顺便确认绘图库已正确安装
import torch       # 提供张量、GPT 模型运行和设备管理功能

# 打印关键依赖版本，便于复现实验以及排查不同版本造成的结果差异
print("thop version:", version("thop"))
print("torch version:", version("torch"))

thop version: 0.1.1-2209072238
torch version: 2.12.0


In [2]:
import torch  # 用于创建输入张量、选择 CPU/GPU，并管理 GPU 显存
from thop import profile  # 运行一次前向传播，并估算模型的 MACs 和参数量

from previous_chapters import GPTModel  # 导入第 4 章前面实现的 GPT 模型


# 四种模型都共用的基础配置
BASE_CONFIG = {
    "vocab_size": 50257,     # 词表大小：输入 token ID 必须在 0～50256 之间
    "context_length": 1024,  # 最大上下文长度：每条输入序列包含 1024 个 token
    "drop_rate": 0.0,        # 关闭 dropout，避免随机丢弃神经元（也便于稳定分析）
    "qkv_bias": True         # Q、K、V 线性映射是否使用偏置项，GPT-2 使用 True
}

# 不同 GPT-2 规模对应的嵌入维度、Transformer 层数和注意力头数
# 名称中的 124M、355M 等是大致参数量；M 表示 million（百万）
model_configs = {
    "gpt-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 有可用的 NVIDIA GPU 时使用 CUDA，否则退回 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 构造一批随机 token ID 作为模型输入，形状为 [批大小 2, 序列长度 1024]
# FLOPs 主要由张量形状决定，因此这里不需要输入真实文本
input_tensor = torch.randint(0, 50257, (2, 1024)).to(device)

# 依次分析四种模型规模
for size in model_configs:
    # 把当前规模特有的 emb_dim、n_layers 和 n_heads 合并到基础配置中
    BASE_CONFIG.update(model_configs[size])
    
    # 根据完整配置创建模型，并使用 bfloat16 降低内存/显存占用
    # 改变数据精度不会改变这里统计的运算次数
    model = GPTModel(BASE_CONFIG).bfloat16()
    model.to(device)  # 将模型参数移动到与输入相同的设备

    # MACs 表示乘加运算次数；一次乘加通常包含一次乘法和一次加法
    # profile 会执行一次模型前向传播，返回估算的 MACs 和参数量 params
    # 注意：THOP 主要统计它能识别的模块；代码中直接使用 @ 完成的注意力矩阵乘法，
    # 以及部分归一化、激活和 softmax 运算可能未被完整统计，所以结果是近似值
    macs, params = profile(model, inputs=(input_tensor,), verbose=False)

    # 按“一次乘法 + 一次加法 = 2 次浮点运算”的惯例，将 MACs 换算为 FLOPs
    # 这里的 FLOPs 指单次前向传播的运算总次数，不是硬件每秒的计算速度（FLOP/s）
    flops = 2 * macs

    # :18 让模型名称占 18 个字符宽度；.1e 用一位小数的科学计数法显示结果
    print(f"{size:18}: {flops:.1e} FLOPS")
    
    # 当前规模统计完成后删除模型，并释放 PyTorch 暂存的 GPU 显存，
    # 为下一种更大的模型腾出空间；在 CPU 环境下 empty_cache() 基本不起作用
    del model
    torch.cuda.empty_cache()

gpt-small (124M)  : 5.1e+11 FLOPS
gpt-medium (355M) : 1.4e+12 FLOPS
gpt-large (774M)  : 3.2e+12 FLOPS
gpt-xl (1558M)    : 6.4e+12 FLOPS
